1. Importing Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

2. Loading the Dataset

In [2]:
df = pd.read_csv('bbc_data.csv')
df["labels"].unique()

array(['entertainment', 'business', 'sport', 'politics', 'tech'],
      dtype=object)

3. Downloading NLTK Resources

In [3]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/vimleshgupta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/vimleshgupta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/vimleshgupta/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

4. Preprocessing the Text

In [4]:
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

5. Applying Preprocessing to Dataset

In [5]:

df['processed_content'] = df['data'].apply(preprocess_text)

df['processed_content'].head()

0    [musicians, tackle, us, red, tape, musicians, ...
1    [desire, number, one, three, prestigious, gram...
2    [rocker, doherty, fight, rock, singer, pete, d...
3    [snicket, tops, us, box, office, chart, film, ...
4    [oceans, twelve, raids, box, office, oceans, t...
Name: processed_content, dtype: object

6. Text Vectorization

In [6]:
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df['processed_content'].apply(' '.join))

6.2. Vectorize Text Using TF-IDF

In [7]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['processed_content'].apply(' '.join))

7. Training our Models

In [8]:
#7.1. Training BoW model
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, df['labels'], test_size=0.2, random_state=42)

nb_model1 = MultinomialNB()

nb_model1.fit(X_train_bow, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [9]:
#7.2. Training TF-IDF model
X_train_tfidf, X_test_tfidf = train_test_split(X_tfidf, test_size=0.2, random_state=42)

nb_model2 = MultinomialNB()

nb_model2.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


8. Comparing Both Models

In [10]:
y_pred_bow = nb_model1.predict(X_test_bow)
print("BoW Model Performance:\n", classification_report(y_test, y_pred_bow))

y_pred_tfidf = nb_model2.predict(X_test_tfidf)
print("TF-IDF Model Performance:\n", classification_report(y_test, y_pred_tfidf))

BoW Model Performance:
                precision    recall  f1-score   support

     business       0.98      0.96      0.97       103
entertainment       1.00      0.98      0.99        84
     politics       0.98      0.99      0.98        80
        sport       1.00      0.99      0.99        98
         tech       0.95      1.00      0.98        80

     accuracy                           0.98       445
    macro avg       0.98      0.98      0.98       445
 weighted avg       0.98      0.98      0.98       445

TF-IDF Model Performance:
                precision    recall  f1-score   support

     business       0.96      0.99      0.98       103
entertainment       1.00      0.95      0.98        84
     politics       0.90      0.97      0.93        80
        sport       0.99      0.99      0.99        98
         tech       1.00      0.93      0.96        80

     accuracy                           0.97       445
    macro avg       0.97      0.97      0.97       445
 weighted

9. Making Prediction

In [11]:
custom_text1 = "Artificial intelligence is revolutionizing the tech industry, with companies racing to develop the next big innovation."

print("Input text: ", custom_text1)

processed_custom_text = ' '.join(preprocess_text(custom_text1))

custom_text_bow = bow_vectorizer.transform([processed_custom_text])
custom_text_tfidf = tfidf_vectorizer.transform([processed_custom_text])

predicted_category_bow = nb_model1.predict(custom_text_bow)
print(f"Predicted Category (BoW): {predicted_category_bow[0]}")

predicted_category_tfidf = nb_model2.predict(custom_text_tfidf)
print(f"Predicted Category (TF-IDF): {predicted_category_tfidf[0]}")

Input text:  Artificial intelligence is revolutionizing the tech industry, with companies racing to develop the next big innovation.
Predicted Category (BoW): tech
Predicted Category (TF-IDF): tech


In [12]:
import pickle

pickle.dump(nb_model1, open("bow_model.pkl", "wb"))
pickle.dump(nb_model2, open("tfidf_model.pkl", "wb"))

pickle.dump(bow_vectorizer, open("bow_vectorizer.pkl", "wb"))
pickle.dump(tfidf_vectorizer, open("tfidf_vectorizer.pkl", "wb"))
